In [1]:
import json
import os
import pandas as pd
import torch
import numpy as np

from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *

load_dotenv()
pd.set_option('display.max_rows', None)

model_ = "gpt-5.1"
log_name_ = "credit_seed52_ratio0.3_synonymous_withLabel"
chunk_size_ = 1

llm = llm_call(model_version = model_, api_key= os.getenv("API_KEY"))
df_new, cases_json = build_event_jsons(log_name = f"./dataset/{log_name_}.csv", chunk_cases = chunk_size_)

## 📝 Pre-processing Phase: Synonymous Label Identification (Step 1)

In this stage, we focus on identifying **Synonymous Labels**—activities that are syntactically different (often substantially) but share the same semantic meaning and represent the exact same business process step.

### 🎯 Objective
The goal is to analyze the unique activity list and extract groups of labels that are semantically equivalent. Unlike distorted labels (which are noise/typos), synonyms consist of valid but inconsistent terminology used by different users or systems.

### 🔍 Detection Criteria (Ontology Rules)
We identify synonyms based on four semantic patterns:
1. **Linguistic & Domain Synonyms**: Different words for the same concept (e.g., `Ship Item` vs. `Dispatch Goods`).
2. **Phrase Variation**: Shared core objects with synonymous verbs (e.g., `Create Invoice` vs. `Generate Invoice`).
3. **Grammatical Transformation**: Changes in parts of speech (e.g., `Approve` vs. `Give Approval`).
4. **Containment & Refinement**: Concise vs. verbose versions (e.g., `Receive signed contract` vs. `Receive contract`).

### ⚖️ Filtering Logic
* **Keep Related Labels**: If Label A and Label B are synonyms, both are kept in the candidate list for further mapping.
* **Discard Isolated Labels**: If an activity is valid but has no semantic matches in the current log, it is removed from the candidate pool.
* **Discard Noise**: Purely distorted (typos) or polluted (IDs) labels are filtered out unless they form a semantic pattern.

### Expected Output
- A JSON object indicating whether synonyms were found and a list of candidate strings for semantic clustering.

In [3]:
activity_list_json = json.dumps(df_new['activity'].unique().tolist(), indent=4, ensure_ascii=False)

SYSTEM_PROMPT_STEP1 = """
You are an expert Process Mining Data Pre-processor.
Your goal is to filter a raw list of activity names based on specific criteria provided in the User Prompt.

### KNOWLEDGE BASE: IMPERFECTION PATTERNS
Use these definitions to identify which labels belong to which category.

1.  **Polluted Labels (Mutable Qualifiers):**
    Labels that share a immutable boiler-plate text but differ due to mutable text (e.g., embedded IDs or codes).
    * **Detection Criteria:**
        * **Long Numeric IDs:** 8+ digits (e.g., `20260122`, `9988776655`).
        * **Mixed Codes:** 6+ alphanumeric chars (e.g., `XJ9281`, `Ref_A1B2C3`).
        * **Delimiters:** Attached via `_`, `-`, `:`, `/`, `#`, `.`, or space.

2.  **Distorted Labels (Character-Level Corruption):**
    Labels containing specific character-level corruptions (typos, OCR faults) of a canonical form. Unlike synonyms, these are "Noise".
    * **Detection Criteria:**
        1.  **Case Mutation:** Identical spelling, different capitalization (e.g., "Open" vs "open" vs "OPEN").
        2.  **Character Omission:** Exactly ONE missing character (e.g., "Invoce" vs "Invoice").
        3.  **Character Insertion:** Exactly ONE extra character (e.g., "Innvoice" vs "Invoice").
        4.  **Character Transposition:** Two adjacent characters swapped (e.g., "Ivnoice" vs "Invoice").
        5.  **Keyboard Proximity:** Exactly ONE character substituted by a QWERTY neighbor (e.g., "Invoicr" vs "Invoice").

3. **Synonymous Labels (Semantic Equivalence):**
   Labels that are syntactically different (often substantially) but share the same semantic meaning and represent the exact same business process step. 
   * **Detection Criteria (Ontology Rules):**
        1. **Linguistic & Domain Synonyms:** Different words representing the same concept within the process context (e.g., "Ship Item" vs "Dispatch Goods", "DrSeen" vs "Medical Assign").
        2. **Phrase Variation (Verb/Object Shift):** Labels sharing a core component (usually the Object) while using synonymous verbs or adjectives (e.g., "Create Invoice" vs "Generate Invoice", "Start instance" vs "Start process").
        3. **Grammatical Transformation:** Changing parts of speech (Noun ↔ Verb) or sentence structure while retaining the core meaning (e.g., "Give approval" vs "Approve", "Conduct analysis" vs "Analyze").
        4. **Containment & Refinement:** One label is a concise or verbose version of the other, often omitting non-essential adjectives, prepositions, or 'online/offline' qualifiers (e.g., "Receive signed contract" vs "Receive contract", "Register for course" vs "Register course").

### GLOBAL INSTRUCTION
- **Role:** Function as a logic engine. Do not assume all imperfections exist.
- **Priority:** The strict filtering logic in the **User Prompt** overrides general definitions here.
- **OUTPUT FORMAT:** Always return valid JSON as requested by the User Prompt.

"""

USER_PROMPT_SYNONYMOUS_STEP1 = f"""
### TASK: Filter Data for 'Synonymous Labels' Candidates

**OBJECTIVE:**
Analyze the provided **INPUT DATA(activity list)** and extract labels related to **Synonymous Labels (Semantic Equivalence)**.
You must **identify and collect** the following components for the output data:
1. All labels that form a **Synonym Group** (two or more labels representing the same process step despite syntactic differences).

**STRICT FILTERING LOGIC:**
1. **Identify Synonym Pairs:** Look for different words or phrases that share the same semantic meaning based on the System Prompt's criteria (Linguistic Synonyms, Phrase Variation, Grammatical Transformation, Containment).
2. **KEEP Related Labels:** If Label A is a semantic synonym of Label B, **KEEP BOTH A and B**. (Unlike Distorted/Polluted, typically all members of a synonym group are valid words, so keep the entire group).
3. **DISCARD Isolated Labels:** If a label is valid but has **NO** semantic synonyms in the provided list, **REMOVE IT**. (e.g., If 'Archive' exists but no synonyms like 'Store' or 'Save' exist, remove 'Archive').
4. **DISCARD Distorted/Polluted:** Remove labels that are purely 'Distorted Labels' (typos) or 'Polluted Labels' (IDs) if they are not part of a 'Synonymous Labels' pattern.

**EDGE CASE HANDLING:**
- If NO Synonymous pairs are found (i.e., all labels are unique/isolated, distorted, or polluted):
    - Return strictly `[]` (with "found": false).
- **Finding NOTHING is a valid result.** Do not force-fit vaguely similar words; strict semantic equivalence is required.

**INPUT DATA:**
{activity_list_json}

***OUTPUT FORMAT GUIDELINES (PERFORMANCE OPTIMIZED)***
Return a JSON Object with two keys:
1. "found": Boolean (true if synonymous labels exist, false otherwise).
2. "data": List of strings.

**Example (Found):**
{{ "found": true, "data": ["Create Invoice", "Generate Invoice", "Make Bill"] }}

**Example (Not Found - SPEED PRIORITY):**
{{ "found": false, "data": [] }}

**CONSTRAINT:**
- Determine the "found" value FIRST. If false, output `[]` for data immediately.
- Output ONLY the JSON.
"""


prompt = [{"role": "system", "content": SYSTEM_PROMPT_STEP1},
          {"role":"user","content": USER_PROMPT_SYNONYMOUS_STEP1}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
synonym_step1 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
synonym_step1_json = json.dumps(synonym_step1['data'], indent=4, ensure_ascii=False)
for k, v in synonym_step1.items():
    print(f"{k}:")
    if isinstance(v, list):  
        if v:  
            print(*v, sep="\n")
        else:  
            print(" (empty list)")
    else: 
        print(f" {v}")
    print() 

found:
 True

data:
Check for completeness
validate completeness
confirm completeness
verify completeness
Perform checks
carry out checks
execute checks
conduct checks
Make decision
take decision
determine outcome
reach decision
Notify accept
notify acceptance
send acceptance notice
issue acceptance
Deliver card
issue card
send card
dispatch card
Request info
request information
request details
ask for information
info received
information received
details received
data received
notify reject
issue rejection
notify rejection
send rejection notice
time out
process timed out
timeout occurred
operation timeout
review request received
review request captured
review request obtained



In [4]:
def get_synonym_context(df: pd.DataFrame,
                        case_col: str = 'case_id',
                        time_col: str = 'timestamp',
                        act_col: str = 'activity',
                        filter_list: set = None):
    df_pm4py = df[[case_col, time_col, act_col]].copy()
    df_pm4py.rename(columns={
        case_col: "case:concept:name",
        time_col: "time:timestamp",
        act_col: "concept:name"
    }, inplace=True)
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"], errors="coerce")
    dfg, start_activities, end_activities = pm4py.discover_dfg(df_pm4py)
    def get_activity_context(activity, dfg_dict):
        predecessors = {k[0]: v for k, v in dfg_dict.items() if k[1] == activity}
        successors = {k[1]: v for k, v in dfg_dict.items() if k[0] == activity}
        total_pred = sum(predecessors.values())
        total_succ = sum(successors.values())
        def format_to_list(dist_dict, total):
            if total == 0: return []
            items = [(k, v/total) for k, v in dist_dict.items() if (v/total) >= 0.05]
            items.sort(key=lambda x: x[1], reverse=True)
            return [k for k, v in items]
        return format_to_list(predecessors, total_pred), format_to_list(successors, total_succ)
    all_activities = sorted(df[act_col].unique())
    flow_data_list = []
    for act in all_activities:
        if filter_list is not None and act not in filter_list:
            continue
        pred, succ = get_activity_context(act, dfg)
        flow_data_list.append({
            'activity': act,
            'predecessors': pred, # 이제 리스트입니다 ['A', 'B']
            'successors': succ    # 이제 리스트입니다 ['C', 'D']
        })
        
    json_flow_context = json.dumps(flow_data_list, indent=2, ensure_ascii=False)
    return json_flow_context


target_activities_set = synonym_step1['data']
synonym_context_json = get_synonym_context(
    df=df_new,          
    filter_list=target_activities_set 
)
print(synonym_context_json)

[
  {
    "activity": "Check for completeness",
    "predecessors": [
      "info received",
      "review request received"
    ],
    "successors": [
      "Request info",
      "Perform checks"
    ]
  },
  {
    "activity": "Deliver card",
    "predecessors": [
      "Notify accept"
    ],
    "successors": []
  },
  {
    "activity": "Make decision",
    "predecessors": [
      "Perform checks"
    ],
    "successors": [
      "notify reject",
      "Notify accept"
    ]
  },
  {
    "activity": "Notify accept",
    "predecessors": [
      "Make decision"
    ],
    "successors": [
      "Deliver card"
    ]
  },
  {
    "activity": "Perform checks",
    "predecessors": [
      "Check for completeness"
    ],
    "successors": [
      "Make decision"
    ]
  },
  {
    "activity": "Request info",
    "predecessors": [
      "Check for completeness"
    ],
    "successors": [
      "info received"
    ]
  },
  {
    "activity": "ask for information",
    "predecessors": [
      "Ch

## 🧩 Synonymous Labels Step 2: Contextual Flow Abstraction

In this stage, we transition from raw activity lists to **Semantic Contexts**. Synonyms often share a similar "structural neighborhood" in a process model. By summarizing the activities that immediately precede or follow a candidate, we can create a high-level signature for each activity.

### 🎯 Objective
1. **Extract Direct Follows Graph (DFG)**: Identify the immediate neighbors (predecessors/successors) of each candidate synonym.
2. **Filter by Significance**: Only keep neighbors that account for at least **5%** of the flow to eliminate infrequent "noise" paths.
3. **LLM-driven Abstraction**: Use a Large Language Model to summarize a list of diverse activity labels (e.g., "Box items", "Wrap package") into a single, descriptive **Process Stage Name** (e.g., "Packaging Phase").

### 🔍 Why Abstraction?
Directly comparing raw predecessor lists is difficult because the predecessors themselves might be synonyms or typos. By abstracting them into "Stages," we create a normalized baseline that allows the LLM to recognize when two syntactically different activities (like "Create Invoice" and "Generate Bill") are performing the same role in the same process phase.

### Expected Output
- `synonym_context_json`: A structured JSON containing the summarized business phases for each candidate synonym's neighborhood.

In [5]:
SYSTEM_PROMPT_SYNONYM_STEP2  = """
You are an expert Process Mining Analyst.
Your goal is to summarize lists of activity labels into a single, descriptive **Process Stage Name**.

### CORE TASK
You will be given an activity and its lists of **Predecessors** (incoming flow) and **Successors** (outgoing flow).
You must analyze the labels in each list and determine the **Common Business Phase** they represent.

### SUMMARIZATION LOGIC (ABSTRACTION)
1. **Identify the Core Action:** Look at the verbs and objects in the list.
2. **Ignore Noise:** Disregard synonyms, typos, and minor variations.
3. **Formulate a Summary:** Create a short, natural language phrase that encapsulates the collective meaning.

### EXAMPLES (Demonstration Only)
- **Input List:** `["Wrap package", "Box items", "Pack goods", "Containerize"]`
- **Output Summary:** "Packaging Phase"

- **Input List:** `["MRI Scan", "X-Ray taken", "Blood test results"]`
- **Output Summary:** "Medical Diagnosis Stage"

- **Input List:** `["Ticket Resolved", "Issue Fixed", "Close Ticket", "Problem Solved"]`
- **Output Summary:** "Ticket Resolution"

### GLOBAL INSTRUCTION
- **Input:** JSON object with `activity`, `predecessors` (list), and `successors` (list).
- **Output:** JSON object where `predecessors` and `successors` are converted to **Strings** (Summaries).
"""

USER_PROMPT_SYNONYM_STEP2 = f"""
### TASK: Summarize Contextual Flow Lists

**OBJECTIVE:**
Analyze the **INPUT DATA**. Replace the list of strings in `predecessors` and `successors` with a **Single Summarized String** describing that process stage.

**STRICT EXECUTION STEPS:**
1. **Iterate** through every activity in the input.
2. **Analyze Predecessors:**
   - Read the list of predecessor labels.
   - Abstract their common meaning into one short phrase (e.g., "Quality Check Phase").
   - **Replace** the list with this string.
3. **Analyze Successors:**
   - Read the list of successor labels.
   - Abstract their common meaning into one short phrase.
   - **Replace** the list with this string.

**INPUT DATA:**
{synonym_context_json}

***OUTPUT FORMAT GUIDELINES***
Return a JSON Object with a single key `"summarized_context"`.
The value must be a list of objects where `predecessors` and `successors` are **STRINGS**, not lists.

**Example Output (Mental Model):**
{{
  "summarized_context": [
    {{
      "activity": "Ship Item",
      "predecessors": "Packaging Phase",     // Was ["Box items", "Wrap package"...]
      "successors": "Delivery Initiation"    // Was ["Truck loaded", "Dispatch"...]
    }},
    {{
      "activity": "Handle Error",
      "predecessors": "System Failure",      // Was ["Crash", "Server Down"...]
      "successors": "Recovery Process"       // Was ["Reboot", "Restart"...]
    }}
  ]
}}

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_SYNONYM_STEP2},
          {"role":"user","content": USER_PROMPT_SYNONYM_STEP2}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
synonym_step2 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
synonym_step2_json = json.dumps(synonym_step2["summarized_context"], indent=2, ensure_ascii=False)
print(synonym_step2_json)

[
  {
    "activity": "Check for completeness",
    "predecessors": "Request or information receipt",
    "successors": "Information clarification or detailed checks"
  },
  {
    "activity": "Deliver card",
    "predecessors": "Acceptance notification to customer",
    "successors": "Process end after card delivery"
  },
  {
    "activity": "Make decision",
    "predecessors": "Application checks and assessment",
    "successors": "Communicate acceptance or rejection"
  },
  {
    "activity": "Notify accept",
    "predecessors": "Final approval decision",
    "successors": "Card preparation and delivery"
  },
  {
    "activity": "Perform checks",
    "predecessors": "Initial completeness verification",
    "successors": "Decision making on application"
  },
  {
    "activity": "Request info",
    "predecessors": "Initial completeness verification",
    "successors": "Additional information received from customer"
  },
  {
    "activity": "ask for information",
    "predecessors": "Com

## 🤝 Synonymous Labels Step 3: Fuzzy Context Clustering with Synonym Boost

This is the final decision-making stage for synonym resolution. We aggregate the abstracted contextual information from Step 2 to perform **Fuzzy Clustering**.

### 🎯 Objective
To group activity labels into clusters where each cluster represents a single, consistent process step. The algorithm moves beyond string matching and evaluates the **core intent** of the activity within the process flow.

### 🔍 Clustering Logic: The "Synonym Boost" Mechanism
The LLM acts as a fuzzy logic engine, evaluating pairs of activities based on two primary factors:

1. **Context Similarity (Base Rule)**:
    - Compares the summarized predecessor and successor strings.
    - If "Data Entry" is compared to "Inputting Details," the model recognizes these as the **same business phase** despite different wording.
2. **Synonym Boost (Tie-Breaker)**:
    - If the activity labels themselves are strong linguistic synonyms (e.g., "Verify" vs. "Check"), the model applies a **leniency bias**.
    - It allows for minor deviations in the context summaries, assuming the functional role is identical.

### ⚖️ Decision Matrix
- **Strong Match**: Both context and label imply the same action $\rightarrow$ **Group**.
- **Synonym Boost**: Labels are synonyms, contexts have minor wording differences $\rightarrow$ **Group**.
- **Mismatch**: Outcomes or contexts clearly diverge (e.g., "Approve" vs. "Reject") $\rightarrow$ **Separate**.

### Expected Output
- A JSON object with a `"clusters"` key containing a **List of Lists** (e.g., `[["Create Order", "Generate Order"], ["Verify", "Check"]]`).
- **Transitivity** is enforced: If A matches B and B matches C, then A, B, and C are clustered together.

In [6]:
SYSTEM_PROMPT_SYNONYM_STEP3 ="""
You are an expert Process Mining Analyst.
Your goal is to cluster activity labels into groups that represent the **Same Process Step**.

### INPUT DATA
You will receive objects with:
1. `activity`: The label.
2. `predecessors`: A summarized string (Input Context).
3. `successors`: A summarized string (Output Context).

### CLUSTERING LOGIC: FUZZY CONTEXT & SYNONYM BOOST
Compare pairs of activities (A and B). Decide if they are the same step based on two factors:

**FACTOR 1: CONTEXT SIMILARITY (The Base Rule)**
- Compare `predecessors_A` vs `predecessors_B` AND `successors_A` vs `successors_B`.
- **Do not look for exact string matches.**
- **Rule:** If the descriptions describe the **Same Business Phase** despite different wording, count it as a MATCH.

**FACTOR 2: LABEL SYNONYM BOOST (The Tie-Breaker)**
- **Rule:** If `activity_A` and `activity_B` are **Linguistic Synonyms** (e.g., "Verify" vs "Check"), you must be **MORE LENIENT** with context matching.
- **Logic:** "If labels imply the same action, allow minor deviations in context phrasing."

### FINAL DECISION MATRIX
1. **Contexts are Semantically Similar:** -> **GROUP**.
2. **Contexts have minor differences BUT Labels are Synonyms:** -> **GROUP** (Synonym Boost).
3. **Contexts are clearly different (Input or Output diverges):** -> **SEPARATE**.

### GLOBAL INSTRUCTION
- **Output:** A JSON object with a single key `"clusters"` containing a **List of Lists**.
- **Constraint:** Ensure Transitivity (A=B, B=C -> A=B=C).
"""

USER_PROMPT_SYNONYM_STEP3 = f"""
### TASK: Fuzzy Context Clustering with Synonym Boost

**OBJECTIVE:**
Group the activities in **INPUT DATA** that represent the same process step.
**Key Instruction:** Be flexible with context descriptions. Focus on the **Core Meaning**.

**STRICT EXECUTION STEPS:**

1. **Analyze Contexts:**
   - Read the natural language summaries.
   - Interpret different phrases describing the same stage as the **SAME** context (e.g., "Data Entry" ≈ "Inputting Data").

2. **Apply Clustering Logic (Use these Mental Models):**
   - **Case 1 (Strong Match):**
     - Context A: "User entering credentials"
     - Context B: "Inputting login details"
     - **Decision:** Contexts mean the same thing. -> **GROUP**.
   - **Case 2 (Synonym Boost):**
     - Labels: "Resolve Ticket" vs "Fix Issue" (Strong Synonyms).
     - Contexts: "Code review" vs "Peer review completed" (Slight wording diff).
     - **Decision:** Labels are synonyms, so ignore the slight context difference. -> **GROUP**.
   - **Case 3 (Mismatch):**
     - Labels: "Approve" vs "Reject".
     - Contexts: "Evaluation" vs "Evaluation". (Context match, but Labels opposite).
     - **Decision:** Clearly different outcome. -> **SEPARATE**.

3. **Apply Transitivity:**
   - Merge all overlapping pairs into final clusters.

**INPUT DATA (Summarized Context):**
{synonym_step2_json}

***OUTPUT FORMAT GUIDELINES***
Return a strict JSON Object with a single key `"clusters"`.

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_SYNONYM_STEP3},
          {"role":"user","content": USER_PROMPT_SYNONYM_STEP3}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
synonym_step3 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
synonym_step3 = synonym_step3['clusters']

print(synonym_step3)

[['Check for completeness', 'confirm completeness', 'validate completeness', 'verify completeness'], ['Perform checks', 'carry out checks', 'conduct checks', 'execute checks'], ['Request info', 'ask for information', 'request details', 'request information'], ['data received', 'details received', 'info received', 'information received'], ['Make decision', 'determine outcome', 'reach decision', 'take decision'], ['Notify accept', 'notify acceptance', 'issue acceptance', 'send acceptance notice'], ['issue rejection', 'notify reject', 'notify rejection', 'send rejection notice'], ['Deliver card', 'dispatch card', 'issue card', 'send card'], ['operation timeout', 'process timed out', 'time out', 'timeout occurred'], ['review request captured', 'review request obtained', 'review request received']]


## 📊 Synonymous Labels Step 4: Frequency-Based Canonical Selection

This is the final arbitration stage where we resolve each synonym cluster into a single **Canonical (Clean) Label** based on empirical evidence from the event log.

### 🎯 Objective
While the LLM identifies which activities belong together semantically, it does not know which label should be the "Standard." This step uses **Frequency Analysis** to select the most prevalent label as the representative for the entire cluster.

### ⚖️ Selection Logic: The "Majority Rule"
1. **Frequency Audit**: For every activity in a cluster, we look up its total occurrence count in the original dataset (`df_new`).
2. **Canonical Selection**: The activity with the **highest frequency** is designated as the `clean_label`. This assumes that the most frequently used term is the intended business standard.
3. **Variant Mapping**: All other activities in the cluster are treated as `variants` and mapped to the clean label.

### Expected Output
- `synonym_step4`: A final mapping dictionary (e.g., `{"Create Order": ["Generate Order", "Order Entry"]}`).
- **Abstraction Report**: A summary showing the transformation from the original canonical label to its associated synonyms.

In [7]:
activity_counts = df_new['activity'].value_counts().to_dict()
synonym_step4 = {}
for cluster in synonym_step3:
    if len(cluster) < 2:
        continue
    clean_label = max(cluster, key=lambda x: activity_counts.get(x, 0))
    variants = sorted([label for label in cluster if label != clean_label])
    synonym_step4[clean_label] = variants
print("\n--- [Process Abstraction Report] ---\n")
for clean, vars in sorted(synonym_step4.items()):
    print(f"Original: '{clean}' -> {vars})")





--- [Process Abstraction Report] ---

Original: 'Check for completeness' -> ['confirm completeness', 'validate completeness', 'verify completeness'])
Original: 'Deliver card' -> ['dispatch card', 'issue card', 'send card'])
Original: 'Make decision' -> ['determine outcome', 'reach decision', 'take decision'])
Original: 'Notify accept' -> ['issue acceptance', 'notify acceptance', 'send acceptance notice'])
Original: 'Perform checks' -> ['carry out checks', 'conduct checks', 'execute checks'])
Original: 'Request info' -> ['ask for information', 'request details', 'request information'])
Original: 'info received' -> ['data received', 'details received', 'information received'])
Original: 'notify reject' -> ['issue rejection', 'notify rejection', 'send rejection notice'])
Original: 'review request received' -> ['review request captured', 'review request obtained'])
Original: 'time out' -> ['operation timeout', 'process timed out', 'timeout occurred'])


In [11]:
df_synonym = df_new[df_new['label'].notna()].copy()
df_synonym['clean_activity'] = df_synonym['label'].str.extract(r'\((.*?)\)')

synonym_answer = (
    df_synonym.groupby('clean_activity')['activity']
    .unique()
    .apply(list)
    .to_dict()
)
print("------------------PREDICTION------------------")
print(json.dumps(synonym_step4, indent=4, ensure_ascii=False))
print("------------------ANSWER------------------")
print(json.dumps(synonym_answer, indent=4, ensure_ascii=False))


------------------PREDICTION------------------
{
    "Check for completeness": [
        "confirm completeness",
        "validate completeness",
        "verify completeness"
    ],
    "Perform checks": [
        "carry out checks",
        "conduct checks",
        "execute checks"
    ],
    "Request info": [
        "ask for information",
        "request details",
        "request information"
    ],
    "info received": [
        "data received",
        "details received",
        "information received"
    ],
    "Make decision": [
        "determine outcome",
        "reach decision",
        "take decision"
    ],
    "Notify accept": [
        "issue acceptance",
        "notify acceptance",
        "send acceptance notice"
    ],
    "notify reject": [
        "issue rejection",
        "notify rejection",
        "send rejection notice"
    ],
    "Deliver card": [
        "dispatch card",
        "issue card",
        "send card"
    ],
    "time out": [
        "operati

In [13]:
def evaluate_synonym_results(answer_dict, predict_dict):
    ans_keys = set(answer_dict.keys())
    pred_keys = set(predict_dict.keys())
    
    tp_keys = ans_keys.intersection(pred_keys)
    fp_keys = pred_keys - ans_keys
    fn_keys = ans_keys - pred_keys
    
    key_precision = len(tp_keys) / len(pred_keys) if pred_keys else 0
    key_recall = len(tp_keys) / len(ans_keys) if ans_keys else 0
    key_f1 = (2 * key_precision * key_recall) / (key_precision + key_recall) if (key_precision + key_recall) else 0
    
    all_v_f1 = []
    all_v_precision = []
    all_v_recall = []

    for key in tp_keys:
        ans_vals = set(answer_dict[key])
        pred_vals = set(predict_dict[key])
        
        tp_v = ans_vals.intersection(pred_vals)
        
        v_prec = len(tp_v) / len(pred_vals) if pred_vals else 0
        v_reca = len(tp_v) / len(ans_vals) if ans_vals else 0
        v_f1 = (2 * v_prec * v_reca) / (v_prec + v_reca) if (v_prec + v_reca) else 0
        
        all_v_precision.append(v_prec)
        all_v_recall.append(v_reca)
        all_v_f1.append(v_f1)
        
    avg_v_precision = sum(all_v_precision) / len(tp_keys) if tp_keys else 0
    avg_v_recall = sum(all_v_recall) / len(tp_keys) if tp_keys else 0
    avg_v_f1 = sum(all_v_f1) / len(tp_keys) if tp_keys else 0

    print("-" * 50)
    print("      [Synonymous Pattern Detection Evaluation Report]")
    print("-" * 50)
    print(f"1. Key Selection (Clean Activity Identification)")
    print(f"   - Precision : {key_precision:.4f}")
    print(f"   - Recall    : {key_recall:.4f}")
    print(f"   - F1-Score  : {key_f1:.4f}")
    print("-" * 50)
    print(f"2. Value Selection (Synonym Variation Mapping Accuracy - Average)")
    print(f"   - Avg Precision : {avg_v_precision:.4f}")
    print(f"   - Avg Recall    : {avg_v_recall:.4f}")
    print(f"   - Avg F1-Score  : {avg_v_f1:.4f}")
    print("-" * 50)
    print(f"   * Analyzed Keys: {len(tp_keys)} matched / {len(ans_keys)} total")
    print("-" * 50)

    return {
        "key_metrics": {"precision": key_precision, "recall": key_recall, "f1": key_f1},
        "value_metrics": {"avg_precision": avg_v_precision, "avg_recall": avg_v_recall, "avg_f1": avg_v_f1}
    }
    
print(evaluate_synonym_results(synonym_answer, synonym_step4))

--------------------------------------------------
      [Synonymous Pattern Detection Evaluation Report]
--------------------------------------------------
1. Key Selection (Clean Activity Identification)
   - Precision : 1.0000
   - Recall    : 1.0000
   - F1-Score  : 1.0000
--------------------------------------------------
2. Value Selection (Synonym Variation Mapping Accuracy - Average)
   - Avg Precision : 1.0000
   - Avg Recall    : 1.0000
   - Avg F1-Score  : 1.0000
--------------------------------------------------
   * Analyzed Keys: 10 matched / 10 total
--------------------------------------------------
{'key_metrics': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}, 'value_metrics': {'avg_precision': 1.0, 'avg_recall': 1.0, 'avg_f1': 1.0}}
